### This jupyter notebook will scraping all company detail of the children and parent's company following this companies using wikipedia


- Procter & Gamble
- PepsiCo
- Nestlé
- Colgate-Palmolive
- L'Oréal
- Coca-Cola Company
- Johnson & Johnson
- Kraft Heinz
- Mondelez International
- Unilever
- Other than above should be group as ‘Unkwon’

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [17]:
def scrape_wikipedia_brand_list(url, parent_name, parent_identifiers):
    forbidden_keywords = [
        "sold", "acquired", "divested", "spun off", "phased out", 
        "discontinued", "merged", "licensed", "taken over", "former"
    ]

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # เลือก Container หลักของเนื้อหา
    content = soup.find('div', class_='mw-parser-output')
    
    brand_lists = []

    if content:
        # วนลูปผ่าน 'ลูก' ทุกตัวที่อยู่ใน div content (Direct Children)
        for element in content.children:
            
            # ตรวจสอบว่าเป็นหัวข้อ (h2, h3) หรือไม่
            if element.name in ['h2', 'h3']:
                text = element.get_text().strip()
                # ถ้าเจอคำว่า References ให้หยุดการทำงานของ Loop ทันที
                if "References" in text or "Notes" in text:
                    break
            
            # ถ้าเจอ tag <ul> ก่อนที่จะถึง References ให้เก็บข้อมูล
            if element.name == 'ul':
                items = element.find_all('li')
                for li in items:
                    # ตัดเลขที่อยู่ใน bracket ออก เช่น [1], [2]
                    brand_raw = re.sub(r'\[.*?\]', '', li.get_text()).strip()
                    
                    # ถ้า li มีโครงสร้างซับซ้อน (เช่น มีหัวข้อแบรนด์แล้วมีรายละเอียดต่อ) 
                    # มักจะเอาแค่บรรทัดแรก หรือก่อนเครื่องหมาย : หรือ -
                    brand_name_clean = brand_raw.split(':')[0].split('–')[0].split('-')[0].strip()
                    
                    brand_lower = brand_name_clean.lower()
                    
                    # 1. Skip if contains forbidden words
                    is_invalid = any(word in brand_lower for word in forbidden_keywords)
                    if is_invalid or len(brand_name_clean) <= 2:
                        continue

                    # 2. Logic to detect Parent vs Child
                    is_parent = any(id_name.lower() == brand_lower for id_name in parent_identifiers)
                    
                    if is_parent:
                        org_type = 'parent_brand'
                    else:
                        org_type = 'child_brand'
                        
                    brand_lists.append({
                        'group_name': brand_name_clean,
                        'parent_company': parent_name,
                        'organization_type': org_type
                    })

    df = pd.DataFrame(brand_lists)
    df = df.drop_duplicates(subset=['group_name'])
    return df

In [19]:
companies = {
    # "Procter & Gamble": ("https://en.wikipedia.org/wiki/List_of_Procter_%26_Gamble_brands", ["procter & gamble", "p&g"]),
    # "PepsiCo": ("https://en.wikipedia.org/wiki/List_of_assets_owned_by_PepsiCo", ["pepsico", "pepsi"]),
    # "Nestlé": ("https://en.wikipedia.org/wiki/List_of_Nestl%C3%A9_brands", ["nestlé", "nestle"]),
    # "Colgate-Palmolive": ("https://en.wikipedia.org/wiki/Colgate-Palmolive#Brands", ["colgate-palmolive"]),
    "L'Oréal": ("https://en.wikipedia.org/wiki/L%27Or%C3%A9al#Brands", ["l'oréal", "l'oreal"]),
    # "Coca-Cola Company": ("https://en.wikipedia.org/wiki/List_of_Coca-Cola_brands", ["coca-cola company", "coca-cola"]),
    # "Johnson & Johnson": ("https://en.wikipedia.org/wiki/Johnson_%26_Johnson#Consumer_health", ["johnson & johnson", "j&j"]),
    # "Kraft Heinz": ("https://en.wikipedia.org/wiki/Kraft_Heinz#Brands", ["kraft heinz", "kraft", "heinz"]),
    # "Mondelez International": ("https://en.wikipedia.org/wiki/Mondelez_International#Brands", ["mondelez international", "mondelez"]),
    # "Unilever": ("https://en.wikipedia.org/wiki/List_of_Unilever_brands", ["unilever"])
}

master_frames = []

for name, (url, ids) in companies.items():
    print(f"Processing: {name}...")
    df_temp = scrape_wikipedia_brand_list(url, name, ids)
    if not df_temp.empty:
        master_frames.append(df_temp)
    time.sleep(1) # Be kind to Wikipedia's servers

# final_df = pd.concat(master_frames, ignore_index=True).drop_duplicates(subset=['group_name', 'parent_company'])

Processing: L'Oréal...


In [20]:
master_frames

[]

In [13]:
def scrape_wikipedia_between_sections(url, parent_name, start_section_text, end_section_text, parent_identifiers):
    headers = {'User-Agent': 'Mozilla/5.0...'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Target the main content area
    content = soup.find('div', class_='mw-parser-output')
    if not content:
        return pd.DataFrame()

    # 1. Find the START and END header tags anywhere in the content
    # We use a lambda to find a header whose text contains our keywords
    start_node = content.find(lambda tag: tag.name in ['h2', 'h3'] and start_section_text.lower() in tag.get_text().lower())
    end_node = content.find(lambda tag: tag.name in ['h2', 'h3'] and end_section_text.lower() in tag.get_text().lower())

    brand_lists = []

    if start_node:
        # 2. Iterate through all siblings following the start_node
        # .find_all_next() gets every single element after the start header
        for element in start_node.find_all_next():
            
            # 3. If we hit the end_node, stop immediately
            if element == end_node:
                break
            
            # 4. Only pull data from <ul> tags found between the two
            if element.name == 'ul':
                items = element.find_all('li')
                for li in items:
                    # Clean the brand name
                    brand_raw = re.sub(r'\[.*?\]', '', li.get_text()).split(':')[0].split('–')[0].strip()
                    
                    if len(brand_raw) <= 2: continue

                    # Classify Parent vs Child
                    brand_lower = brand_raw.lower()
                    is_parent = any(id_name.lower() in brand_lower for id_name in parent_identifiers)
                    
                    brand_lists.append({
                        'group_name': brand_raw,
                        'parent_company': parent_name,
                        'organization_type': 'parent_brand' if is_parent else 'child_brand'
                    })

    df = pd.DataFrame(brand_lists).drop_duplicates(subset=['group_name'])
    return df

# --- Try it with L'Oréal ---
df_loreal = scrape_wikipedia_between_sections(
    url="https://en.wikipedia.org/wiki/L%27Or%C3%A9al",
    parent_name="L'Oréal",
    start_section_text="Brand portfolio", # The specific section header
    end_section_text="Marketing",       # The next section header where we stop
    parent_identifiers=["L'Oréal", "L'Oreal"]
)

print(df_loreal)

                                           group_name parent_company  \
0              Dark and Lovely Professional Expertise        L'Oréal   
1   Essie, founded in 1981 and acquired by L'Oreal...        L'Oréal   
2                                             Garnier        L'Oréal   
3                                       La Provençale        L'Oréal   
4                                       L'Oréal Paris        L'Oréal   
..                                                ...            ...   
58                                            Logocos        L'Oréal   
59                                          Sanoflore        L'Oréal   
60                                 Skinbetter Science        L'Oréal   
61       Skinceuticals Advanced Professional Skincare        L'Oréal   
62                            Vichy's Vichy cosmetics        L'Oréal   

   organization_type  
0        child_brand  
1       parent_brand  
2        child_brand  
3        child_brand  
4       parent_brand